In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import deel.torchlip as tl

from common import *
from torchvision import transforms
from torch.optim import SGD

from opacus import PrivacyEngine

In [ ]:
# Experiment
n_epochs = 50
n_exp = 10
class_names = ["clear-day", "snow-day", "clear-evening", "snow-evening"]

# Training
lr = 0.01
momentum = 0.9
weight_decay = 1e-4
bs_train = 128
bs_test = 64
train_ratio = 0.8
schedule = True

# Privacy
private = True

# Data / quantum
qpc = 10
b_size = 2 ** (qpc // 2)
dim1_blocks = 2
dim2_blocks = 3
dim1 = dim1_blocks * b_size
dim2 = dim2_blocks * b_size
input_shape = (dim1, dim2)

loader_dir = "./loaders/4class_2000/qasm"
img_dir = "./loaders/original_data_4class_2000"

# Model-specific
k_lip = 1.5

In [ ]:
class LipschitzCNN(nn.Module):
    def __init__(self, num_classes=4, k_lip=1.0):
        super().__init__()
        self.k_lip = float(k_lip)

        self.conv1 = tl.SpectralConv2d(1, 4, kernel_size=3, padding=1)
        self.act1 = tl.GroupSort2()
        self.pool1 = tl.ScaledAvgPool2d(2, 2)

        self.conv2 = tl.SpectralConv2d(4, 4, kernel_size=1)
        self.act2 = tl.GroupSort2()
        self.pool2 = tl.ScaledAvgPool2d(2, 2)

        self.conv3 = tl.SpectralConv2d(4, 4, kernel_size=1)
        self.act3 = tl.GroupSort2()
        self.pool3 = tl.ScaledAvgPool2d(2, 2)

        self.conv4 = tl.SpectralConv2d(4, 4, kernel_size=1)
        self.act4 = tl.GroupSort2()
        self.pool4 = tl.ScaledAvgPool2d(2, 2)

        self.conv5 = tl.SpectralConv2d(4, 4, kernel_size=1)
        self.act5 = tl.GroupSort2()
        self.pool5 = tl.ScaledAvgPool2d(2, 2)

        self.conv6 = tl.SpectralConv2d(4, 4, kernel_size=1)
        self.act6 = tl.GroupSort2()

        self.conv7 = tl.SpectralConv2d(4, 4, kernel_size=1)
        self.act7 = tl.GroupSort2()

        self.fc1 = tl.SpectralLinear(4 * 2 * 3, 4)
        self.actfc = tl.GroupSort2()
        self.fc2 = tl.SpectralLinear(4, num_classes)

    def forward(self, x):
        x = self.pool1(self.act1(self.conv1(x)))
        x = self.pool2(self.act2(self.conv2(x)))
        x = self.pool3(self.act3(self.conv3(x)))
        x = self.pool4(self.act4(self.conv4(x)))
        x = self.pool5(self.act5(self.conv5(x)))
        x = self.act6(self.conv6(x))
        x = self.act7(self.conv7(x))
        x = x.view(x.size(0), -1)
        x = self.actfc(self.fc1(x))
        x = self.fc2(x)
        return self.k_lip * x

In [ ]:
def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())


@torch.no_grad()
def count_flops(
    model: nn.Module,
    input_shape=(1, 1, 128, 256),
    *,
    count_batchnorm: bool = True,
    bn_mode: str = "train_like",
) -> int:
    flops = 0
    hooks = []

    def conv_hook(m, inp, out):
        nonlocal flops
        B, Cout, Hout, Wout = out.shape
        Cin_eff = m.in_channels // m.groups
        N = Cin_eff * m.kernel_size[0] * m.kernel_size[1]
        flops += B * Cout * Hout * Wout * (2 * N)

    def linear_hook(m, inp, out):
        nonlocal flops
        B = inp[0].shape[0]
        flops += B * m.out_features * (2 * m.in_features)

    def bn_hook(m, inp, out):
        nonlocal flops
        if not count_batchnorm or bn_mode == "none":
            return
        numel = out.numel()
        if bn_mode == "eval":
            flops += 4 * numel
        elif bn_mode == "train_like":
            flops += 10 * numel
        else:
            raise ValueError(f"Unknown bn_mode={bn_mode}.")

    for m in model.modules():
        if isinstance(m, nn.Conv2d):
            hooks.append(m.register_forward_hook(conv_hook))
        elif isinstance(m, nn.Linear):
            hooks.append(m.register_forward_hook(linear_hook))
        elif isinstance(m, nn.BatchNorm2d):
            hooks.append(m.register_forward_hook(bn_hook))

    model.eval()
    _ = model(torch.randn(*input_shape))

    for h in hooks:
        h.remove()

    return int(flops)

In [ ]:
quantum_dataset = CustomLoaderDataset(
    path=loader_dir,
    class_names=class_names,
    dim1_blocks=dim1_blocks,
    dim2_blocks=dim2_blocks,
)

filter_names = ["_".join(el[-1].split("_")[-2:]) for el in quantum_dataset.data]

classical_dataset = CustomImageDataset(
    path=img_dir,
    class_names=class_names,
    filter_names=filter_names,
    transform=transforms.Compose([
        transforms.Resize(input_shape),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
    ]),
)

train_loader, test_loader = get_data_loaders(
    dataset=classical_dataset,
    bs_train=bs_train,
    bs_test=bs_test,
    train_ratio=train_ratio,
)

In [ ]:
all_train_losses = []
all_test_losses = []
all_test_accs = []

for exp in range(n_exp):
    model = LipschitzCNN(num_classes=len(class_names), k_lip=k_lip)
    opt = SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)

    if private:
        privacy_engine = PrivacyEngine()
        model, opt, train_loader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=opt,
            data_loader=train_loader,
            epochs=n_epochs,
            clipping="flat",
            grad_sample_mode="hooks",
            max_grad_norm=1.5,
            target_epsilon=10.0,
            target_delta=1e-5,
        )
    else:
        privacy_engine = None

    scheduler = None
    if schedule:
        scheduler = optim.lr_scheduler.OneCycleLR(
            opt, lr, epochs=n_epochs, steps_per_epoch=len(train_loader)
        )

    train_losses, test_losses, test_accs = train(
        model, opt, sched=scheduler,
        n_epochs=n_epochs,
        train_loader=train_loader,
        test_loader=test_loader,
        eng=privacy_engine,
    )

    all_train_losses.append(train_losses)
    all_test_losses.append(test_losses)
    all_test_accs.append(test_accs)